# Agentic Workflow

This notebook turns the baseline into an inspectable state machine. Each node updates `AgentState`, appends to the trace, and makes one workflow decision explicit.

## Learning goals

- Understand the difference between a static pipeline and a stateful agent.
- Inspect the shared `AgentState` object step by step.
- See how planning, tools, verification, and fallback alter behavior.
- Read execution traces as debugging artifacts rather than opaque logs.


## Concept explanation

As in the first notebook, we start by verifying the Python executable. This is especially useful on DGX or remote Jupyter setups where it is easy to connect the wrong kernel by accident.


In [ ]:
import sys
print(sys.executable)


This setup cell adds the project root to `sys.path` and imports the public workflow nodes that mirror the architecture diagram. The notebook never reimplements core logic; it only orchestrates functions from `src/`.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.ingestion import build_demo_index
from src.state import AgentState, create_initial_state
from src.utils import display_trace
from src.workflow import (
    classify_query_node,
    decide_tools_node,
    fallback_or_finalize_node,
    make_plan_node,
    normalize_query_node,
    retrieve_docs_node,
    run_tools_node,
    run_workflow,
    synthesize_answer_node,
    verify_grounding_node,
)

pd.set_option('display.max_colwidth', 140)
retriever = build_demo_index(persist=False)


### What is an agentic workflow

A pipeline always follows the same recipe. An agentic workflow still uses explicit steps, but it allows those steps to inspect state and adapt behavior. That is why this project keeps the workflow deterministic and inspectable instead of hiding everything in one model call.

### Difference between pipeline and agent

The important distinction here is not autonomy for its own sake. It is the ability to classify the query, select tools only when useful, and refuse unsupported answers when the evidence is weak.

### Stateful agent design

`AgentState` is the shared memory for the notebook. Reviewing its fields makes the architecture concrete before we start mutating it.

## Implementation


In [ ]:
state_schema = pd.DataFrame(
    {
        'field': list(AgentState.__annotations__.keys()),
        'type_hint': [str(value) for value in AgentState.__annotations__.values()],
    }
)
state_schema


### Node concept

We begin with a fresh state and run only the normalization node. This shows the simplest possible state transition: one input field becomes one cleaned output field, and the trace records it.


In [ ]:
state = create_initial_state('How many days are in the pilot window?')
normalize_query_node(state)
pd.Series(state)


### Query classification

The classifier turns a raw text question into a small control signal. In this design the control signal is the query type plus a flag that says whether tools are likely to help.


In [ ]:
classify_query_node(state)
pd.Series(
    {
        'normalized_query': state['normalized_query'],
        'query_type': state['query_type'],
        'requires_tools': state['requires_tools'],
    }
)


### Planning

Planning does not mean open-ended chain-of-thought here. It means selecting a task template that is easy to explain in an interview and easy to verify in code.


In [ ]:
make_plan_node(state)
pd.DataFrame({'planned_step': state['plan']})


### Retrieval

Once the workflow knows what kind of question it is handling, it retrieves the most relevant document chunks. We inspect both the scores and the source files because retrieval quality shapes everything that follows.


In [ ]:
retrieve_docs_node(state, retriever=retriever, top_k=4)
pd.DataFrame(state['retrieved_docs'])[['chunk_id', 'source', 'score', 'text']]


### Tools

The tool stage has two parts: decide which tools would help, then run them locally. For the pilot-window question the workflow parses dates and computes a day difference instead of trying to infer the duration from prose alone.


In [ ]:
decide_tools_node(state)
run_tools_node(state)
pd.DataFrame(state['tool_outputs'])


### Answer synthesis

Synthesis combines the retrieved sentences and any tool outputs into a draft answer with citations. This draft is still provisional because the verifier has not judged its support yet.


In [ ]:
synthesize_answer_node(state)
print(state['draft_answer'])
pd.DataFrame(state['citations'])


### Verification

The verifier checks whether each answer sentence is supported by the retrieved evidence and tool outputs. The point is not to be perfect; it is to make grounding explicit and inspectable.


In [ ]:
verify_grounding_node(state)
pd.Series(state['verification_result'].to_dict())


### Fallback strategy

The fallback node converts verification into a product decision: finalize the answer or abstain. This is where the workflow protects itself from confident-but-unsupported responses.


In [ ]:
fallback_or_finalize_node(state)
pd.Series({'final_status': state['final_status'], 'final_answer': state['final_answer']})


## Experiment

Running the full workflow end to end is useful because it confirms the manual walk-through matches the production path. We also test an out-of-scope query to see abstention in action.


In [ ]:
happy_path = run_workflow('How many days are in the pilot window?', retriever=retriever)
out_of_scope = run_workflow('Who is the current CEO of the company?', retriever=retriever)
comparison = pd.DataFrame(
    [
        {
            'question': happy_path['user_query'],
            'final_status': happy_path['final_status'],
            'final_answer': happy_path['final_answer'],
        },
        {
            'question': out_of_scope['user_query'],
            'final_status': out_of_scope['final_status'],
            'final_answer': out_of_scope['final_answer'],
        },
    ]
)
comparison


## Result analysis

Execution traces are the fastest way to understand why the workflow behaved the way it did. A good trace should show the node name, the important inputs, and the outputs that moved the state forward.


In [ ]:
display_trace(happy_path['trace'])


## Takeaways

- Agentic behavior here comes from explicit state transitions, not hidden magic.
- Planning and tools help on questions the baseline cannot resolve well.
- Verification and fallback make abstention a first-class outcome.
- The trace gives you a debugging surface you can explain in a portfolio walkthrough.
